# Data Segmentation EDA

Checks record counts by city and property type to determine whether separate ML models per segment are feasible.

**Data source:** `Data/processed/listings_all_cities.parquet` (42 823 listings across Madrid, Barcelona, Málaga)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DATA_PATH = "../Data/processed/listings_all_cities.parquet"
MIN_SAMPLES = 500  # threshold below which a separate model is not viable

df = pd.read_parquet(DATA_PATH)
print(f"Total records: {len(df):,}")
print(f"Columns: {df.shape[1]}")

## 1. Records by City

In [ ]:
city_counts = df["city"].value_counts().rename_axis("city").reset_index(name="count")
city_counts["pct"] = (city_counts["count"] / len(df) * 100).round(1)
city_counts["viable_model"] = city_counts["count"] >= MIN_SAMPLES

print(city_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(city_counts["city"], city_counts["count"], color="steelblue")
ax.bar_label(bars, labels=[f"{v:,}" for v in city_counts["count"]], padding=4)
ax.axvline(MIN_SAMPLES, color="red", linestyle="--", linewidth=1, label=f"Min threshold ({MIN_SAMPLES:,})")
ax.set_xlabel("Number of listings")
ax.set_title("Records by city")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Records by Property Type

In [ ]:
# Use the standardised column when available; fall back to raw property_type
prop_col = "property_type_std" if "property_type_std" in df.columns else "property_type"

prop_counts = (
    df[prop_col]
    .value_counts()
    .rename_axis("property_type")
    .reset_index(name="count")
)
prop_counts["pct"] = (prop_counts["count"] / len(df) * 100).round(1)
prop_counts["viable_model"] = prop_counts["count"] >= MIN_SAMPLES

print(prop_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(4, len(prop_counts) * 0.4)))
colors = ["steelblue" if v else "lightcoral" for v in prop_counts["viable_model"]]
bars = ax.barh(prop_counts["property_type"], prop_counts["count"], color=colors)
ax.bar_label(bars, labels=[f"{v:,}" for v in prop_counts["count"]], padding=4, fontsize=8)
ax.axvline(MIN_SAMPLES, color="red", linestyle="--", linewidth=1, label=f"Min threshold ({MIN_SAMPLES:,})")
ax.set_xlabel("Number of listings")
ax.set_title(f"Records by property type ('{prop_col}')")
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cross-tab: City × Property Type

In [ ]:
crosstab = pd.crosstab(df[prop_col], df["city"], margins=True, margins_name="TOTAL")
crosstab.index.name = "property_type"
print(crosstab.to_string())

## 4. Model Feasibility: City-Level Separate Models

In [ ]:
feasibility_city = city_counts.copy()
feasibility_city["verdict"] = feasibility_city["viable_model"].map(
    {True: "✓ viable", False: "✗ too few"}
)
print("=== City-level model feasibility ===")
print(feasibility_city[["city", "count", "pct", "verdict"]].to_string(index=False))

## 5. Model Feasibility: City × Property Type Segments

In [ ]:
seg = (
    df.groupby(["city", prop_col])
    .size()
    .reset_index(name="count")
    .sort_values(["city", "count"], ascending=[True, False])
)
seg["pct_of_city"] = seg.groupby("city")["count"].transform(lambda x: (x / x.sum() * 100).round(1))
seg["viable_model"] = seg["count"] >= MIN_SAMPLES
seg["verdict"] = seg["viable_model"].map({True: "✓ viable", False: "✗ too few"})

print("=== City × property type feasibility ===")
print(seg.to_string(index=False))

In [ ]:
n_viable = seg["viable_model"].sum()
n_total = len(seg)
print(f"\nViable segments (>= {MIN_SAMPLES} records): {n_viable} / {n_total}")
print("\nViable segments:")
print(seg[seg["viable_model"]][["city", prop_col, "count", "pct_of_city"]].to_string(index=False))
print("\nNon-viable segments (would need pooling):")
print(seg[~seg["viable_model"]][["city", prop_col, "count", "pct_of_city"]].to_string(index=False))

## 6. Summary Heat-map

In [ ]:
pivot = seg.pivot(index=prop_col, columns="city", values="count").fillna(0).astype(int)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(8, max(4, len(pivot) * 0.45)))
im = ax.imshow(pivot.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        text_color = "white" if val > pivot.values.max() * 0.6 else "black"
        ax.text(j, i, f"{val:,}", ha="center", va="center", fontsize=8, color=text_color)
ax.set_title(f"Listings per city × property type (red dashed = {MIN_SAMPLES:,} threshold)")
plt.colorbar(im, ax=ax, label="count")
plt.tight_layout()
plt.show()